# Tech Challenge Fase 2
## 04.2 — Bronze Streaming

Lê os JSONs com Structured Streaming e preserva os eventos com metadados técnicos.

## 1. Imports e schema

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("event_type", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("co_uf", StringType(), True),
    StructField("sg_uf", StringType(), True),
    StructField("co_municipio", StringType(), True),
    StructField("no_municipio", StringType(), True),
    StructField("indicador", StringType(), False),
    StructField("valor", DoubleType(), True),
    StructField("origem", StringType(), True),
    StructField("event_version", IntegerType(), False)
])

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"
config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))
STREAMING_PATH = config["paths"]["streaming_path"]
INPUT_PATH = f"{STREAMING_PATH}/input"
BRONZE_PATH = f"{STREAMING_PATH}/bronze_streaming"
CHECKPOINT_PATH = f"{STREAMING_PATH}/checkpoint/bronze"

## 3. Leitura e persistência

In [0]:
df_stream = (
    spark.readStream
    .schema(event_schema)
    .option("maxFilesPerTrigger", 1)
    .json(INPUT_PATH)
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_file_name",
        F.col("_metadata.file_name")
    )
    .withColumn(
        "_source_file_size",
        F.col("_metadata.file_size")
    )
    .withColumn(
        "_source_file_modification_time",
        F.col("_metadata.file_modification_time")
    )
    .withColumn(
        "_pipeline_step",
        F.lit("bronze_streaming")
    )
)

query = (
    df_stream
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )
    .trigger(availableNow=True)
    .start(BRONZE_PATH)
)

query.awaitTermination()

print("Bronze Streaming concluída.")

## 4. Validação

In [0]:
df = spark.read.parquet(BRONZE_PATH)
print("Registros:", df.count())
display(df.orderBy(F.col("event_timestamp").desc()).limit(20))